# SUSPEKT · YOLO-Modelle weitertrainieren (Google Colab)

Dieses Notebook trainiert ein bestehendes SUSPEKT-YOLO-Modell (Ultralytics, `.pt`)
mit neu gelabelten Bildern aus **Label Studio** weiter und exportiert das Ergebnis
als `.pt` und `.onnx` — fertig für Webseite und Jetson-Demonstrator.

**Voraussetzungen**

1. **GPU-Laufzeit aktivieren:** Menü *Laufzeit → Laufzeittyp ändern → T4 GPU*.
2. In Google Drive existiert dieser Ordner (einmalig anlegen):
   ```
   MyDrive/SUSPEKT/
   ├── modelle/           ← aktuelles Basismodell, z. B. 251104_real_Y12m_detect_29cls.pt
   ├── datensaetze/       ← yolo_dataset.zip  (Export aus Label Studio, siehe training/README.md)
   │                        optional: alt_datensatz.zip (historischer Datensatz für den Replay-Mix)
   └── modelle_neu/       ← hier landen die fertig trainierten Modelle (legt das Notebook an)
   ```
3. Das Datensatz-Zip wurde mit `training/export_yolo_dataset.py --zip` erzeugt
   (empfohlen) **oder** ist ein nativer YOLO-Export aus Label Studio — beides wird
   unterstützt.

**Ablauf:** Zellen einfach von oben nach unten ausführen (▶︎ bzw. `Umschalt+Enter`).
Nur die Konfigurations-Zelle (Schritt 2) muss ggf. angepasst werden.


## Schritt 1 · GPU prüfen und Ultralytics installieren

In [ ]:
# Zeigt die zugewiesene GPU. Erscheint ein Fehler, ist keine GPU-Laufzeit aktiv
# (Menü: Laufzeit -> Laufzeittyp aendern -> T4 GPU) — dann Laufzeit neu starten.
!nvidia-smi

In [ ]:
%pip install -q ultralytics

## Schritt 2 · Google Drive einbinden und konfigurieren

**Hier bei Bedarf anpassen** (Dateinamen des Basismodells und des Datensatz-Zips).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ------------------------- KONFIGURATION -------------------------
BASIS_ORDNER   = '/content/drive/MyDrive/SUSPEKT'

# Das Modell, das weitertrainiert werden soll:
BASIS_MODELL   = f'{BASIS_ORDNER}/modelle/251104_real_Y12m_detect_29cls.pt'

# Der neue Datensatz (Export aus Label Studio):
DATENSATZ_ZIP  = f'{BASIS_ORDNER}/datensaetze/yolo_dataset.zip'

# Optional, aber dringend empfohlen (Replay-Mix gegen "Vergessen"):
# Zip mit dem historischen Trainingsdatensatz, '' = deaktiviert.
ALT_DATENSATZ_ZIP = ''  # z. B. f'{BASIS_ORDNER}/datensaetze/alt_datensatz.zip'

# Trainings-Einstellungen (konservative Werte fuers Fine-Tuning):
EPOCHEN  = 50
IMGSZ    = 640      # wie das Originaltraining; 320 ist nur die Jetson-Inferenzgroesse
BATCH    = 16
LERNRATE = 0.001    # niedrig, damit vorhandenes Wissen erhalten bleibt
GEDULD   = 20       # Early Stopping: Abbruch nach 20 Epochen ohne Verbesserung

# Namenspraefix fuer das fertige Modell (Versionsschema YYMMDD_<praefix>...):
MODELL_ZWECK = 'komponenten'   # z. B. 'komponenten' oder 'nubs'
# ------------------------------------------------------------------

import os
assert os.path.exists(BASIS_MODELL), f'Basismodell nicht gefunden: {BASIS_MODELL}'
assert os.path.exists(DATENSATZ_ZIP), f'Datensatz-Zip nicht gefunden: {DATENSATZ_ZIP}'
print('Konfiguration OK.')

## Schritt 3 · Datensatz entpacken

Entpackt das Zip, mischt optional den Alt-Datensatz bei (Replay-Mix) und legt
bei einem flachen Label-Studio-Export automatisch einen 80/20-Split an.

In [ ]:
import random
import shutil
import zipfile
from pathlib import Path

DATA = Path('/content/dataset')
if DATA.exists():
    shutil.rmtree(DATA)
DATA.mkdir(parents=True)


def entpacke(zip_pfad, ziel):
    ziel = Path(ziel)
    ziel.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_pfad) as z:
        z.extractall(ziel)
    # Falls das Zip einen einzelnen Wurzelordner enthaelt: eine Ebene hochziehen.
    eintraege = [p for p in ziel.iterdir() if not p.name.startswith('.')]
    if len(eintraege) == 1 and eintraege[0].is_dir():
        for p in eintraege[0].iterdir():
            shutil.move(str(p), ziel)
        eintraege[0].rmdir()
    return ziel


entpacke(DATENSATZ_ZIP, DATA)

# Flacher Export (images/ + labels/ ohne train/val)? Dann selbst splitten.
if not (DATA / 'images' / 'train').exists():
    bilder = sorted(p for p in (DATA / 'images').glob('*.*'))
    random.Random(42).shuffle(bilder)
    n_train = int(len(bilder) * 0.8)
    for split, teil in (('train', bilder[:n_train]), ('val', bilder[n_train:])):
        (DATA / 'images' / split).mkdir(parents=True, exist_ok=True)
        (DATA / 'labels' / split).mkdir(parents=True, exist_ok=True)
        for bild in teil:
            label = DATA / 'labels' / (bild.stem + '.txt')
            shutil.move(str(bild), DATA / 'images' / split / bild.name)
            if label.exists():
                shutil.move(str(label), DATA / 'labels' / split / label.name)
    print(f'Flacher Export erkannt: {n_train} train / {len(bilder) - n_train} val gesplittet.')

# Replay-Mix: Alt-Datensatz zusaetzlich einmischen (verhindert Vergessen).
if ALT_DATENSATZ_ZIP:
    ALT = entpacke(ALT_DATENSATZ_ZIP, '/content/dataset_alt')
    kopiert = 0
    for split in ('train', 'val'):
        for bild in (ALT / 'images' / split).glob('*.*'):
            ziel_name = f'alt_{bild.name}'
            shutil.copy(bild, DATA / 'images' / split / ziel_name)
            label = ALT / 'labels' / split / (bild.stem + '.txt')
            if label.exists():
                shutil.copy(label, DATA / 'labels' / split / f'alt_{bild.stem}.txt')
            kopiert += 1
    print(f'Replay-Mix: {kopiert} Alt-Bilder beigemischt.')
else:
    print('ACHTUNG: Kein Alt-Datensatz konfiguriert. Bei kleinen neuen Datensaetzen')
    print('kann das Modell bereits Gelerntes vergessen (siehe Konzept, Abschnitt 7).')

for split in ('train', 'val'):
    n = len(list((DATA / 'images' / split).glob('*.*')))
    print(f'{split}: {n} Bilder')

## Schritt 4 · Klassen mit dem Basismodell abgleichen

Die Klassen-Indizes müssen exakt denen des Basismodells entsprechen, sonst
„verlernt“ das Modell beim Fine-Tuning. Abweichende Reihenfolgen werden
automatisch korrigiert; unbekannte Klassen brechen bewusst ab.

In [ ]:
import yaml
from ultralytics import YOLO

basismodell = YOLO(BASIS_MODELL)
modell_klassen = [basismodell.names[i] for i in sorted(basismodell.names)]
print(f'Basismodell: {len(modell_klassen)} Klassen')

# Klassenliste des Datensatzes ermitteln (data.yaml oder classes.txt).
if (DATA / 'data.yaml').exists():
    cfg = yaml.safe_load((DATA / 'data.yaml').read_text())
    ds_klassen = cfg['names']
    if isinstance(ds_klassen, dict):
        ds_klassen = [ds_klassen[k] for k in sorted(ds_klassen)]
elif (DATA / 'classes.txt').exists():
    ds_klassen = [z.strip() for z in (DATA / 'classes.txt').read_text().splitlines() if z.strip()]
else:
    raise SystemExit('Weder data.yaml noch classes.txt im Datensatz gefunden.')

unbekannt = set(ds_klassen) - set(modell_klassen)
if unbekannt:
    raise SystemExit(
        f'Abbruch: Der Datensatz enthaelt Klassen, die das Basismodell nicht kennt: '
        f'{sorted(unbekannt)}. Entweder Tippfehler in Label Studio korrigieren oder '
        f'bewusst mit erweiterter Klassenliste von einem Basismodell ohne Vortraining starten.')

if ds_klassen != modell_klassen:
    mapping = {i: modell_klassen.index(name) for i, name in enumerate(ds_klassen)}
    for txt in (DATA / 'labels').rglob('*.txt'):
        zeilen = []
        for zeile in txt.read_text().splitlines():
            teile = zeile.split()
            if teile:
                teile[0] = str(mapping[int(teile[0])])
                zeilen.append(' '.join(teile))
        txt.write_text('\n'.join(zeilen) + ('\n' if zeilen else ''))
    print('Klassen-Indizes an das Basismodell angepasst.')
else:
    print('Klassenreihenfolge stimmt bereits mit dem Basismodell ueberein.')

FINALE_YAML = DATA / 'data_final.yaml'
FINALE_YAML.write_text(yaml.safe_dump({
    'path': str(DATA),
    'train': 'images/train',
    'val': 'images/val',
    'nc': len(modell_klassen),
    'names': modell_klassen,
}, allow_unicode=True, sort_keys=False))
print(f'Trainings-Konfiguration geschrieben: {FINALE_YAML}')

## Schritt 5 · Training starten

Fine-Tuning ab dem Basismodell. Richtwert: einige hundert Bilder, 50 Epochen
≈ 1–2 h auf einer T4. Der Fortschritt wird live angezeigt.

> **Colab-Abbruch?** Sitzungen sind zeitlich begrenzt. Nach einem Abbruch:
> Schritte 1–4 erneut ausführen und in dieser Zelle die `resume`-Zeile
> einkommentieren, dann setzt das Training am letzten Checkpoint fort.

In [ ]:
ergebnis = basismodell.train(
    data=str(FINALE_YAML),
    epochs=EPOCHEN,
    imgsz=IMGSZ,
    batch=BATCH,
    lr0=LERNRATE,
    patience=GEDULD,
    project='/content/runs',
    name='weitertraining',
    exist_ok=True,
)

# Nach einem Sitzungs-Abbruch stattdessen fortsetzen:
# ergebnis = YOLO('/content/runs/weitertraining/weights/last.pt').train(resume=True)

BEST = '/content/runs/weitertraining/weights/best.pt'
print(f'Bestes Modell: {BEST}')

## Schritt 6 · Qualitäts-Gate: neues Modell gegen das alte vergleichen

Beide Modelle werden auf **demselben Validierungs-Set** gemessen.
**Regel: Nur ausrollen, wenn das neue Modell nicht schlechter ist.**

In [ ]:
from ultralytics import YOLO

modell_neu = YOLO(BEST)
modell_alt = YOLO(BASIS_MODELL)

print('— Validierung: NEUES Modell —')
metrik_neu = modell_neu.val(data=str(FINALE_YAML), imgsz=IMGSZ, verbose=False)
print('— Validierung: ALTES Modell —')
metrik_alt = modell_alt.val(data=str(FINALE_YAML), imgsz=IMGSZ, verbose=False)

zeilen = [
    ('mAP50-95', metrik_alt.box.map, metrik_neu.box.map),
    ('mAP50', metrik_alt.box.map50, metrik_neu.box.map50),
    ('Precision', metrik_alt.box.mp, metrik_neu.box.mp),
    ('Recall', metrik_alt.box.mr, metrik_neu.box.mr),
]
print(f"\n{'Metrik':<12}{'alt':>8}{'neu':>8}")
for name, alt, neu in zeilen:
    print(f'{name:<12}{alt:>8.3f}{neu:>8.3f}')

if metrik_neu.box.map >= metrik_alt.box.map:
    print('\n✅ Qualitaets-Gate BESTANDEN: Das neue Modell darf ausgerollt werden.')
else:
    print('\n❌ Qualitaets-Gate NICHT bestanden: Neues Modell ist schlechter.')
    print('   Moegliche Ursachen: zu wenige/unausgewogene neue Bilder, fehlender')
    print('   Replay-Mix, Labelfehler. NICHT ausrollen — Daten pruefen.')

In [ ]:
# Confusion Matrix des neuen Modells anzeigen (welche Klassen werden verwechselt?)
from IPython.display import Image as IPyImage, display
from pathlib import Path

cm = sorted(Path(str(metrik_neu.save_dir)).glob('confusion_matrix*.png'))
if cm:
    display(IPyImage(filename=str(cm[0]), width=900))

## Schritt 7 · Stichprobe: Vorhersagen auf Validierungsbildern ansehen

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

beispiele = sorted((DATA / 'images' / 'val').glob('*.*'))[:6]
vorhersagen = modell_neu.predict([str(p) for p in beispiele], conf=0.4, verbose=False)

fig, achsen = plt.subplots(2, 3, figsize=(18, 10))
for achse, ergebnis in zip(achsen.flat, vorhersagen):
    achse.imshow(ergebnis.plot()[:, :, ::-1])
    achse.axis('off')
plt.tight_layout()
plt.show()

## Schritt 8 · Exportieren und in Drive sichern

Erzeugt `<YYMMDD>_<zweck>_weitertrainiert.pt` und `.onnx` in
`MyDrive/SUSPEKT/modelle_neu/`.

In [ ]:
import shutil
from datetime import datetime
from pathlib import Path

version = datetime.now().strftime('%y%m%d')
name = f'{version}_{MODELL_ZWECK}_weitertrainiert'

onnx_pfad = modell_neu.export(format='onnx', imgsz=IMGSZ, simplify=True)

ziel = Path(BASIS_ORDNER) / 'modelle_neu'
ziel.mkdir(parents=True, exist_ok=True)
shutil.copy(BEST, ziel / f'{name}.pt')
shutil.copy(onnx_pfad, ziel / f'{name}.onnx')

# Trainingskurven mit sichern (fuer das Modell-Logbuch).
kurven = Path('/content/runs/weitertraining/results.png')
if kurven.exists():
    shutil.copy(kurven, ziel / f'{name}_trainingskurven.png')

print(f'Gesichert in {ziel}:')
for datei in sorted(ziel.glob(f'{name}*')):
    print(f'  {datei.name}')

## Schritt 9 · Deployment (außerhalb von Colab)

**Webseite (SUSPEKT):**
1. `<name>.pt` aus Drive nach `webapp/model/` kopieren.
2. In der `.env` den Eintrag `MODEL_NAME` auf den neuen Dateinamen stellen.
3. App/Container neu starten und mit 2–3 Referenzbildern gegentesten.
4. Rollback: `MODEL_NAME` zurückstellen — das alte Modell bleibt liegen.

**Demonstrator (Jetson):**
1. `<name>.pt` nach `models/` im Demonstrator-Repo kopieren.
2. `YOLO_SOURCE_*` in `src/demonstrator/config/settings.py` anpassen.
3. **Auf dem Jetson** `tools/convert_models.sh` ausführen — TensorRT-Engines
   sind gerätespezifisch und müssen auf der Zielhardware gebaut werden
   (dort wird auch die 320er-Inferenzgröße/FP16 gesetzt).
4. Demonstrator starten und Live-Erkennung prüfen.

**Modell-Logbuch pflegen:** Datum, Datensatzgröße, mAP alt/neu, wer trainiert
hat, Besonderheiten. Vorlage siehe Schulungsunterlagen
(`docs/retraining/schulung.md`).


---
## Optional · Datensatz aus Roboflow statt Label Studio

Falls der Datensatz in Roboflow gepflegt wird, kann er direkt geladen werden —
danach ab **Schritt 4** normal weitermachen (`DATA` zeigt dann auf den
Roboflow-Ordner).


In [ ]:
# %pip install -q roboflow
# from roboflow import Roboflow
# from pathlib import Path
#
# rf = Roboflow(api_key='DEIN_ROBOFLOW_API_KEY')
# projekt = rf.workspace('DEIN_WORKSPACE').project('DEIN_PROJEKT')
# datensatz = projekt.version(1).download('yolov11', location='/content/dataset_roboflow')
# DATA = Path(datensatz.location)
# print('Roboflow-Datensatz geladen:', DATA)